# 03 - Walk-forward analysis

This is the main comparison. Every method uses the same trailing window, one-month lag, rebalance schedule, long-only rule and cost model. Turnover is measured against the weights just before each trade.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from robust_dm_factor_allocation import (
    METHODS,
    ExactERCCapError,
    active_metrics,
    annualized_return,
    load_config,
    load_monthly_returns,
    performance_metrics,
    plot_wealth,
    plot_weights,
    walk_forward_backtest,
)

In [ ]:
working_directory = Path.cwd().resolve()
repository_root = (
    working_directory.parent if working_directory.name == "notebooks" else working_directory
)
config = load_config(repository_root / "config" / "config.yaml")
monthly_returns = load_monthly_returns(
    config.data_file,
    factor_columns=config.factor_columns,
    market_column=config.benchmark,
    missing=config.missing_policy,
)
index_metadata = pd.read_csv(
    repository_root / "data" / "metadata" / "index_metadata.csv",
    parse_dates=["launch_date"],
)

tables_directory = repository_root / "results" / "tables"
weights_directory = repository_root / "results" / "portfolio_weights"
figures_directory = repository_root / "results" / "figures"
for directory in (tables_directory, weights_directory, figures_directory):
    directory.mkdir(parents=True, exist_ok=True)

## Five allocation rules

Equal weight is the baseline. Inverse volatility, sample GMV, OAS-GMV and ERC only use data available before the execution month. ERC is marked as unavailable if its exact weights break the configured cap.

In [ ]:
backtests = {}
status_rows = []
for method in METHODS:
    try:
        result = walk_forward_backtest(
            monthly_returns,
            method=method,
            lookback=config.lookback_months,
            rebalance_months=config.rebalance_frequency_months,
            lag=config.lag_months,
            max_weight=config.maximum_weight,
            transaction_cost_bps=config.transaction_cost_bps,
            factor_columns=config.factor_columns,
            market_column=config.benchmark,
            missing=config.missing_policy,
        )
    except ExactERCCapError:
        status_rows.append(
            {
                "method": method,
                "status": "infeasible_exact_erc",
                "reason": "The exact ERC solution breaches the weight cap.",
            }
        )
        continue
    backtests[method] = result
    status_rows.append({"method": method, "status": "ok", "reason": None})

method_status = pd.DataFrame(status_rows).set_index("method").reindex(METHODS)
if "oas_gmv" not in backtests:
    raise RuntimeError("The primary OAS-GMV backtest is unavailable.")
method_status

In [ ]:
for method, result in backtests.items():
    targets = result["target_weights"].dropna(how="all")
    if not targets.sum(axis=1).sub(1).abs().lt(1e-9).all():
        raise RuntimeError(f"{method} produced targets that do not sum to one.")
    if (targets < -1e-10).any().any():
        raise RuntimeError(f"{method} produced a short position.")
    if (targets > config.maximum_weight + 1e-10).any().any():
        raise RuntimeError(f"{method} exceeded the configured cap.")
    rebalances = result["returns"].loc[lambda frame: frame["rebalance"]]
    if not (rebalances.index > rebalances["estimation_end"]).all():
        raise RuntimeError(f"{method} failed the estimation-lag audit.")

## Full and post-launch results

The post-launch period starts with the first full monthly return after the latest factor launch. Its first estimates can still use provider-backtested history in the trailing window.

In [ ]:
feasible_methods = [method for method in METHODS if method in backtests]
walk_forward_returns = pd.DataFrame(
    {method: backtests[method]["returns"]["net_return"] for method in feasible_methods}
)
walk_forward_gross_returns = pd.DataFrame(
    {method: backtests[method]["returns"]["gross_return"] for method in feasible_methods}
)
walk_forward_turnover = pd.DataFrame(
    {method: backtests[method]["returns"]["turnover"] for method in feasible_methods}
)
reference_returns = backtests["oas_gmv"]["returns"]
walk_forward_returns[config.benchmark] = reference_returns["benchmark_return"]
walk_forward_gross_returns[config.benchmark] = reference_returns["benchmark_return"]

walk_forward_diagnostics = reference_returns[["rebalance", "estimation_end"]].copy()
walk_forward_diagnostics["estimation_window_start"] = walk_forward_diagnostics[
    "estimation_end"
] - pd.offsets.MonthEnd(config.lookback_months - 1)
walk_forward_diagnostics["lookback_months"] = config.lookback_months
walk_forward_diagnostics["lag_months"] = config.lag_months

for _method, result in backtests.items():
    schedule = result["returns"][["rebalance", "estimation_end"]]
    if not schedule.equals(reference_returns[["rebalance", "estimation_end"]]):
        raise RuntimeError("Walk-forward schedules differ across methods.")

In [ ]:
benchmark_status = pd.DataFrame(
    {"status": ["benchmark"], "reason": [None]},
    index=[config.benchmark],
)
status_table = pd.concat([method_status, benchmark_status])
launch_dates = index_metadata.set_index("series_id")["launch_date"]
factor_launch_dates = launch_dates.reindex(config.factor_columns)
if factor_launch_dates.isna().any():
    raise ValueError("A configured factor is missing its launch date.")
latest_factor_launch = factor_launch_dates.max()
first_post_launch_return = latest_factor_launch.to_period("M").to_timestamp(
    "M"
) + pd.offsets.MonthEnd(1)
if first_post_launch_return > walk_forward_returns.index.max():
    raise ValueError("No post-launch evaluation months are available.")
evaluation_periods = {
    "historical_walk_forward": walk_forward_returns.index.min(),
    "post_launch_outcome": max(first_post_launch_return, walk_forward_returns.index.min()),
}
period_definitions = pd.Series(evaluation_periods, name="start").to_frame()
period_definitions.index.name = "period"

summary_rows = []
for period, start_date in evaluation_periods.items():
    net_period = walk_forward_returns.loc[start_date:]
    gross_period = walk_forward_gross_returns.loc[net_period.index]
    turnover_period = walk_forward_turnover.loc[net_period.index]
    for series_name in [*METHODS, config.benchmark]:
        status = status_table.loc[series_name]
        row = {
            "period": period,
            "series": series_name,
            "status": status["status"],
            "reason": status["reason"],
            "start": net_period.index.min(),
            "end": net_period.index.max(),
            "months": len(net_period),
        }
        if series_name not in net_period:
            summary_rows.append(row)
            continue
        row.update(
            performance_metrics(
                net_period[series_name],
                periods_per_year=config.periods_per_year,
            ).to_dict()
        )
        if series_name != config.benchmark:
            row.update(
                active_metrics(
                    net_period[series_name],
                    net_period[config.benchmark],
                    periods_per_year=config.periods_per_year,
                ).to_dict()
            )
            row["annualized_cost_drag"] = annualized_return(
                gross_period[series_name],
                periods_per_year=config.periods_per_year,
            ) - annualized_return(
                net_period[series_name],
                periods_per_year=config.periods_per_year,
            )
            row["annualized_turnover"] = (
                turnover_period[series_name].mean() * config.periods_per_year
            )
        summary_rows.append(row)

walk_forward_summary = pd.DataFrame(summary_rows).set_index(["period", "series"])
walk_forward_summary

In [ ]:
method_status.to_csv(tables_directory / "walk_forward_method_status.csv")
walk_forward_returns.to_csv(tables_directory / "walk_forward_returns.csv")
walk_forward_gross_returns.to_csv(tables_directory / "walk_forward_gross_returns.csv")
walk_forward_turnover.to_csv(tables_directory / "walk_forward_turnover.csv")
walk_forward_diagnostics.to_csv(tables_directory / "walk_forward_diagnostics.csv")
period_definitions.to_csv(tables_directory / "walk_forward_periods.csv")
walk_forward_summary.to_csv(tables_directory / "walk_forward_performance.csv")

for method, result in backtests.items():
    result["weights"].to_csv(weights_directory / f"{method}_applied_weights.csv")
    result["target_weights"].to_csv(weights_directory / f"{method}_target_weights.csv")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
plot_wealth(walk_forward_returns, ax=ax, log_scale=True)
ax.set_title("Walk-forward wealth after transaction costs")
fig.tight_layout()
fig.savefig(figures_directory / "walk_forward_wealth.png", dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 6))
plot_weights(backtests["oas_gmv"]["weights"], ax=ax)
ax.set_title("OAS-GMV weights applied each month")
fig.tight_layout()
fig.savefig(figures_directory / "walk_forward_oas_gmv_weights.png", dpi=160)
plt.close(fig)

## Reading the results

First check whether each method ran. Then compare net performance, turnover and cost drag. MSCI World is the benchmark, not an allocation sleeve. Notebook 04 tests other OAS-GMV settings.